# Latent Phase 1 -- KV hand-off (latent/)

One session: load Qwen3-4B (fp16, sdpa) + the v2.1 adapter, run the **exactness checks** (the gate),
the **value test** (A works out a value, B must reuse it) and the **thinker test** (the thinker's reasoning
as memory vs the 600-char PLAN, on the 20 g6 items: an answer task and a verify task `v_*`), plus the
adapter-mismatch check. Gate and mismatch numbers are saved to `checks.json` / `mismatch.json`. See `latent/README.md`.
Grading is local (`python latent/score_local.py <unzipped latent_out>`); answer keys never reach Kaggle.

**Before you start:** GPU **T4** (one is used), **Internet on**, inputs:
1. the **v2.1 adapters** dataset (folder `v2.1-lora-full-lr0.0002-s1/best`);
2. the **probe 4 prompts** dataset (`main_prompts.jsonl`);
3. optional: the **g6 s0 think-first output** containing `thinker.jsonl` (else traces are generated here, slower).

**Push `latent/` first:** the notebook resets to `origin/main`.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/arkapravopal04/Swarm_neural_nets.git'
REPO_DIR = '/kaggle/working/Swarm_neural_nets'

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{cmd} failed: {r.stdout} {r.stderr}')
    return r

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run(['git', 'clone', REPO_URL, REPO_DIR])
run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'])
run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR + '/latent')
print('synced to:', run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s']).stdout.strip())

In [ ]:
!pip -q install transformers==5.22.2 peft==0.21.2 accelerate==1.15.0

In [ ]:
# CPU smoke of the kv plumbing on a tiny random Qwen3 (seconds). Stop here if it fails.
r = subprocess.run([sys.executable, 'latent/smoke_cpu.py'], capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-4000:])
assert r.returncode == 0, 'latent/smoke_cpu.py FAILED: fix kv.py before using the GPU session'

In [ ]:
import glob, json

RUN_CHECKS = True
RUN_VALUE = True
RUN_THINKER = True
RUN_MISMATCH = True
N_THINKER_ITEMS = 20          # the first N of the 20 g6 ids

hits = [h for h in glob.glob('/kaggle/input/**/v2.1-lora-full-lr0.0002-s1/best/adapter_config.json', recursive=True)
        if '/adapters_pack/' not in h]
assert len(hits) == 1, f'attach the v2.1 adapters dataset (found {len(hits)})'
ADAPTER_PATH = os.path.dirname(hits[0])
found = glob.glob('/kaggle/input/**/main_prompts.jsonl', recursive=True)
assert len(found) == 1, f'attach the probe 4 prompts dataset (found {found})'
PROMPTS = found[0]
tj = glob.glob('/kaggle/input/**/thinker.jsonl', recursive=True)
THINKER_JSONL = tj[0] if tj else None
OUT = '/kaggle/working/latent_out'
os.makedirs(OUT, exist_ok=True)
print('adapter', ADAPTER_PATH, '| prompts', PROMPTS, '| thinker.jsonl', THINKER_JSONL, '->', OUT)

In [ ]:
import torch
import peft.tuners.lora.torchao as _lora_torchao
_lora_torchao.is_torchao_available = lambda: False  # Kaggle's torchao 0.10 vs this peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B', dtype=torch.float16,
                                             attn_implementation='sdpa', device_map={'': 0})
model = PeftModel.from_pretrained(model, ADAPTER_PATH).eval()
print(f'VRAM used: {torch.cuda.memory_allocated() / 2**30:.2f} GiB')

In [ ]:
import kv, checks

A_TEXT = ('<|im_start|>user\nA tank holds 840 litres and drains at 35 litres per minute. A second pump adds '
          '12 litres per minute. How long until the tank is empty?<|im_end|>\n<|im_start|>assistant\n'
          'The net outflow is the drain rate minus the inflow: 35 - 12 = 23 litres per minute. The tank starts '
          'with 840 litres, so the time to empty is 840 / 23 minutes. Doing the division: 23 x 36 = 828, which '
          'leaves 12 litres, and 12 / 23 is about 0.5217. So the time is about 36.52 minutes. As a check, '
          '36.52 x 23 = 839.96, which is 840 to rounding. In seconds that is 36 minutes and about 31 seconds. '
          'The key value to remember is 36.52 minutes; the rates 35 and 12 were only needed to get the net '
          'outflow of 23 litres per minute.<|im_end|>\n')
B_TEXT = ('<|im_start|>user\nFrom your work above, how many minutes until the tank is empty?<|im_end|>\n'
          '<|im_start|>assistant\n<think>\n\n</think>\n\n')
dev = next(model.parameters()).device
a_ids = torch.tensor([tok.encode(A_TEXT, add_special_tokens=False)], device=dev)
b_ids = torch.tensor([tok.encode(B_TEXT, add_special_tokens=False)], device=dev)
print('A tokens', a_ids.shape[1], '| B tokens', b_ids.shape[1])
if RUN_CHECKS:
    with torch.inference_mode(), model.disable_adapter():
        res = checks.run_checks(model, a_ids, b_ids)
    print(res)
    with open(os.path.join(OUT, 'checks.json'), 'w') as f:
        json.dump(res, f, indent=1)
    assert res['passed'], 'exactness checks FAILED: every later number in this notebook is meaningless until kv.py is fixed'

In [ ]:
import value_test
if RUN_VALUE:
    value_res = value_test.run_value_test(model, tok, OUT)

In [ ]:
import thinker_test
if RUN_THINKER:
    items = thinker_test.load_items(PROMPTS, thinker_test.G6_IDS[:N_THINKER_ITEMS])
    traces = thinker_test.get_traces(model, tok, items, OUT, thinker_jsonl=THINKER_JSONL)
    thinker_test.run_thinker_test(model, tok, items, traces, OUT)
    if RUN_MISMATCH:
        thinker_test.mismatch_check(model, tok, items, traces, n=3, out_dir=OUT)

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/latent_out', 'zip', OUT)
print('wrote /kaggle/working/latent_out.zip')
print(f'peak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB')